# Solar filament instance segmentation — complete reproduction code
Shengkai Zhu.

All project code needed to reproduce submissions **56835266** (match)
and **56835269** (expected IoU) is included in the cells below. No cell
invokes a Python script, imports `filaments`, writes a Python module, or
loads source code from another file. The notebook uses the supplied
weights, model configurations and fitted rescorers, plus standard libraries.

This is the complete **inference and verification** pipeline for the
trained ens5c ensemble. Training is not rerun. Install `requirements.txt`
in a fresh Python 3.12 environment and open this notebook from the repository
root. Supply the official competition data after accepting its terms.
No test annotations are read. Reference predictions are opened only after
fresh inference, to measure reproduction agreement, not test accuracy.

## 1. Configuration and dependencies
`DEVICE="cuda"` and `LIMIT=0` reproduce the full 180-image test set.
For a CPU smoke run use `DEVICE="cpu"` and `LIMIT=1`; partial outputs
have a `smoke_` prefix and must not be submitted. Set `FILAMENT_DATA_ROOT`
to the directory containing `train/` and `test/`, or edit `DATA` below.
The optional training cache is unnecessary for inference and is disabled
by default. Use one GPU job at a time and allow at least 6 GB free RAM.

In [ ]:
from pathlib import Path
import os
import json
import hashlib
import contextlib
from collections import defaultdict
from dataclasses import dataclass, field, asdict

os.environ['OMP_NUM_THREADS'] = '2'
os.environ['OPENBLAS_NUM_THREADS'] = '2'

import cv2
import joblib
import numpy as np
import pandas as pd
import torch
from torch import nn
import segmentation_models_pytorch as smp
from scipy import ndimage as ndi
from skimage.segmentation import watershed
from pycocotools import mask as mask_util

ROOT = Path.cwd().resolve()
assert (ROOT / 'weights/manifest.json').is_file(), 'Run from repository root'
DEVICE = os.environ.get('FILAMENT_DEVICE', 'cuda')
LIMIT = int(os.environ.get('FILAMENT_LIMIT', '0'))
assert 0 <= LIMIT <= 180
BUILD_TRAINING_CACHE = False
OUTPUTS = Path(os.environ.get('FILAMENT_OUTPUT_DIR', str(ROOT / 'outputs')))
DATA = Path(os.environ.get('FILAMENT_DATA_ROOT',
                          str(ROOT / 'data/MAGFiLO_1.0_Kaggle_2026'))).resolve()
if not DATA.exists() and 'FILAMENT_DATA_ROOT' not in os.environ:
    matches = list(Path('/kaggle/input').rglob(
        'MAGFiLO_1.0_Annotations_kaggle2026_train.json'))
    assert len(matches) == 1, 'Set FILAMENT_DATA_ROOT to the official dataset'
    DATA = matches[0].parent.parent
TRAIN_JSON = DATA / 'train/MAGFiLO_1.0_Annotations_kaggle2026_train.json'
TRAIN_IMAGES = DATA / 'train/train_images'
TEST_IMAGES = DATA / 'test/test_images'
CACHE = ROOT / 'data/cache'
assert TRAIN_JSON.is_file() and TEST_IMAGES.is_dir(), DATA
MEMBERS = ['r50-long', 'cnx-s-bce', 'cnx-s-long', 'r50-consensus', 'cnx-s-consensus']
SIZE = 2048
MEAN, STD = 0.45, 0.225
FLIPS = [(), (-1,), (-2,), (-2, -1)]
OBSERVATORIES = 'BCLMTU'
MIN_CANDIDATE = 50
torch.set_num_threads(2)
print('Data:', DATA, '| Device:', DEVICE, '| Images:', LIMIT or 180)

## 2. Reconstruct and verify the five checkpoints
The chunks contain the exact fine-tuned parameters. Verify chunk and
checkpoint SHA-256 hashes; no external pretrained weights are downloaded.

In [ ]:
def sha(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda:f.read(4*1024**2), b""):
            h.update(block)
    return h.hexdigest()

In [ ]:
def restore_weights():
    records = json.loads((ROOT / "weights/manifest.json").read_text())
    for record in records:
        target = ROOT / record["target"]
        if target.exists() and sha(target) == record["sha256"]:
            print("Verified", record["target"], flush=True)
            continue
        target.parent.mkdir(parents=True, exist_ok=True)
        tmp = target.with_suffix(".tmp")
        with tmp.open("wb") as out:
            for part in record["parts"]:
                source = ROOT / part["path"]
                assert source.stat().st_size == part["size"] and sha(source) == part["sha256"], source
                with source.open("rb") as f:
                    for block in iter(lambda:f.read(4*1024**2), b""):
                        out.write(block)
        assert tmp.stat().st_size == record["size"] and sha(tmp) == record["sha256"], target
        tmp.replace(target)
        print("Restored", record["target"], flush=True)

In [ ]:
restore_weights()

## 3. Images, COCO masks and observation split
Images remain grayscale at 2048 × 2048. Split by observation, so all
annotations of the same image stay together. The original 601/106 split
and all 180 test IDs are verified before inference.

In [ ]:
def encode_counts(mask: np.ndarray) -> str:
    """Máscara (H, W) bool/uint8 -> string `counts` para la columna segmentation_rle."""
    rle = mask_util.encode(np.asfortranarray(mask.astype(np.uint8)))
    return rle["counts"].decode("ascii")


def decode_counts(counts: str, size: int = SIZE) -> np.ndarray:
    return mask_util.decode({"size": [size, size], "counts": counts.encode("ascii")}).astype(bool)


def polygon_to_mask(segmentation: list[list[float]], h: int = SIZE, w: int = SIZE) -> np.ndarray:
    rles = mask_util.frPyObjects(segmentation, h, w)
    return mask_util.decode(mask_util.merge(rles)).astype(bool)

In [ ]:
@dataclass
class Observation:
    stem: str  # p. ej. 20140609195854Bh
    annotations: dict[str, list[list[float]]] = field(default_factory=dict)  # image id -> polígonos

    @property
    def observatory(self) -> str:
        return self.stem[-2:]


def load_observations(path: Path = TRAIN_JSON) -> dict[str, Observation]:
    coco = json.loads(path.read_text())
    obs: dict[str, Observation] = {}
    id_to_stem = {}
    for im in coco["images"]:
        stem = Path(im["file_name"]).stem
        id_to_stem[im["id"]] = stem
        obs.setdefault(stem, Observation(stem)).annotations[im["id"]] = []
    for a in coco["annotations"]:
        obs[id_to_stem[a["image_id"]]].annotations[a["image_id"]].append(a["segmentation"][0])
    return obs


def split(stems: list[str], val_frac: float = 0.15, seed: int = 0) -> tuple[list[str], list[str]]:
    """Split por observación, estratificado por observatorio."""
    rng = np.random.default_rng(seed)
    by_obs = defaultdict(list)
    for s in sorted(stems):
        by_obs[s[-2:]].append(s)
    train, val = [], []
    for group in by_obs.values():
        group = list(rng.permutation(group))
        k = round(len(group) * val_frac)
        val += group[:k]
        train += group[k:]
    return sorted(train), sorted(val)


def rasterize(polygons: list[list[float]]) -> np.ndarray:
    """Polígonos -> mapa de etiquetas uint8. Mismo criterio que pycocotools (verificado en scripts/prepare_cache)."""

    label = np.zeros((SIZE, SIZE), np.uint8)
    for k, poly in enumerate(polygons, start=1):
        label[polygon_to_mask([poly])] = k
    return label


def read_image(path: Path) -> np.ndarray:
    im = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    assert im is not None and im.shape == (SIZE, SIZE), path
    return im

In [ ]:
def cache_image(path: Path) -> None:
    out = CACHE / "images" / f"{path.stem}.npy"
    if not out.exists():
        np.save(out, read_image(path))


def cache_label(item: tuple[str, list]) -> tuple[str, list]:
    ann_id, polygons = item
    out = CACHE / "labels" / f"{ann_id}.npy"
    lab = rasterize(polygons)
    if not out.exists():
        np.save(out, lab)
    cents = []
    for k in range(1, len(polygons) + 1):
        ys, xs = np.nonzero(lab == k)
        if len(ys):
            cents.append([float(ys.mean()), float(xs.mean())])
    return ann_id, cents

In [ ]:
observations = load_observations()
train_stems, val_stems = split(list(observations))
test_paths = sorted(TEST_IMAGES.glob('*.jpeg'))
original = json.loads((ROOT / 'metadata/original_split.json').read_text())
generated = dict(train=train_stems, val=val_stems, test=[p.stem for p in test_paths])
assert all(generated[key] == original[key] for key in ('train', 'val', 'test'))
assert len(test_paths) == 180
assert not set(train_stems) & set(val_stems)
if BUILD_TRAINING_CACHE:
    (CACHE / 'images').mkdir(parents=True, exist_ok=True)
    (CACHE / 'labels').mkdir(parents=True, exist_ok=True)
    for path in sorted(TRAIN_IMAGES.glob('*.jpeg')) + test_paths:
        cache_image(path)
    items = [(ann, polygons) for obs in observations.values()
             for ann, polygons in obs.annotations.items()]
    generated['centroids'] = dict(cache_label(item) for item in items)
    generated['annotations'] = {s: sorted(o.annotations) for s, o in observations.items()}
    (CACHE / 'index.json').write_text(json.dumps(generated))
print('Verified split:', len(train_stems), len(val_stems), len(test_paths))

## 4. U-Net and four-flip test-time augmentation
Two ResNet50 and three ConvNeXt-S encoders produce foreground and contact
logits. The exact architecture comes from each supplied configuration.
Each member averages four flips; probabilities are converted to float16
as in the accepted pipeline, then float32 for the five-member average.

In [ ]:
def pick(device: str | None = None) -> str:
    if device and device != "auto":
        return device
    if torch.cuda.is_available():
        return "cuda"
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        return "mps"
    return "cpu"


def autocast(device: str):
    kind = device.split(":")[0]
    if kind == "cuda":
        return torch.autocast("cuda", dtype=torch.float16)
    if kind == "mps":
        return torch.autocast("mps", dtype=torch.bfloat16)
    return contextlib.nullcontext()

In [ ]:
class FilamentNet(nn.Module):
    def __init__(self, arch: str = "Unet", encoder: str = "resnet34", weights: str | None = "imagenet",
                 decoder_attention_type: str | None = None, no_maxpool: bool = False):
        super().__init__()
        kwargs = {"decoder_attention_type": decoder_attention_type} if decoder_attention_type is not None else {}
        self.net = getattr(smp, arch)(encoder_name=encoder, encoder_weights=weights, in_channels=1, classes=2,
                                      **kwargs)
        if no_maxpool:
            if not encoder.startswith("resnet") or not isinstance(self.net.encoder.maxpool, nn.MaxPool2d):
                raise ValueError("no_maxpool solo está disponible para encoders ResNet")
            self.net.encoder.maxpool = nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net((x - MEAN) / STD)


def build(cfg: dict, pretrained: bool = True) -> FilamentNet:
    model = FilamentNet(cfg["arch"], cfg["encoder"], "imagenet" if pretrained else None,
                       cfg.get("decoder_attention_type"), cfg.get("no_maxpool", False))
    # Opt-in for attention encoders: native-resolution tiles avoid quadratic
    # attention memory at 2048px. Old configurations retain full-image inference.
    model.inference_tile = cfg.get("inference_tile", 0)
    model.inference_overlap = cfg.get("inference_overlap", 128)
    return model

In [ ]:
@torch.no_grad()
@torch.inference_mode()
def predict(model: torch.nn.Module, image: np.ndarray, tta: bool = True, device: str | None = None) -> np.ndarray:
    """image uint8 (H, W) -> probabilidades float16 (2, H, W): filamento, borde.

    inference_mode: sin él, cada pasada a 2048² guardaba el grafo de autograd (varios GB de pico); en WSL/WDDM ese
    pico no da OOM sino que pagina VRAM a RAM y el entrenamiento que evalúa queda ~5x más lento (2026-09-26)."""
    device = pick(device)
    tile = getattr(model, "inference_tile", 0)
    if tile:
        return predict_tiled(model, image, tile, getattr(model, "inference_overlap", 128), tta, device)
    x = torch.from_numpy(np.asarray(image, dtype=np.float32) / 255.0)[None, None].to(device).contiguous(memory_format=torch.channels_last)
    acc = torch.zeros((2, *image.shape), device=device)
    flips = FLIPS if tta else FLIPS[:1]
    with autocast(device):
        for dims in flips:
            xi = torch.flip(x, dims) if dims else x
            p = torch.sigmoid(model(xi).float())
            acc += (torch.flip(p, dims) if dims else p)[0]
    return (acc / len(flips)).half().cpu().numpy()


@torch.inference_mode()
def predict_tiled(model, image, tile=512, overlap=128, tta=True, device="cpu"):
    """Blend overlapping native-resolution tiles; invert each TTA before blending.

    Accumulate on CPU to bound GPU memory independently of full image size.
    Positive Hann weights avoid seams and uncovered image borders.
    """
    if tile <= 0 or tile % 32 or not 0 <= overlap < tile:
        raise ValueError("tile must be a positive multiple of 32, 0 <= overlap < tile")
    height, width = image.shape
    padded = np.pad(image, ((0, max(0, tile-height)), (0, max(0, tile-width))), mode="edge")
    h, w = padded.shape
    def starts(n):
        return sorted(set(range(0, n-tile+1, tile-overlap)) | {n-tile})
    axis = np.maximum(np.hanning(tile).astype(np.float32), 0.05)
    weight = axis[:, None] * axis[None, :]
    acc = np.zeros((2, h, w), np.float32)
    norm = np.zeros((h, w), np.float32)
    flips = FLIPS if tta else FLIPS[:1]
    for y in starts(h):
        for x in starts(w):
            patch = np.ascontiguousarray(padded[y:y+tile, x:x+tile], dtype=np.float32) / 255
            inp = torch.from_numpy(patch)[None, None].to(device)
            total = torch.zeros((1, 2, tile, tile), device=device)
            with autocast(pick(device)):
                for dims in flips:
                    prob = model(torch.flip(inp, dims) if dims else inp).float().sigmoid()
                    total += torch.flip(prob, dims) if dims else prob
            acc[:, y:y+tile, x:x+tile] += total[0].cpu().numpy() / len(flips) * weight
            norm[y:y+tile, x:x+tile] += weight
    return (acc[:, :height, :width] / norm[:height, :width]).astype(np.float16)

## 5. Contact-aware watershed and fragment merging
Threshold the foreground, seed away from contact regions, apply watershed,
and merge nearby fragments according to the fitted configuration.

In [ ]:
@dataclass(frozen=True)
class PostCfg:
    t_fg: float = 0.5
    t_b: float = 0.5
    split: bool = True
    merge_dist: int = 0
    min_area: int = 150
    min_score: float = 0.0

    def as_dict(self) -> dict:
        return asdict(self)


def instances(prob: np.ndarray, cfg: PostCfg) -> np.ndarray:
    """prob (2, H, W) -> mapa de etiquetas int32 (0 = fondo)."""
    pf = prob[0].astype(np.float32)
    mask = pf > cfg.t_fg
    comps, n_comp = ndi.label(mask)
    if n_comp == 0:
        return comps
    if cfg.split:
        seeds, _ = ndi.label(mask & (prob[1] < cfg.t_b))
        lab = watershed(-pf, seeds, mask=mask)
        # componentes sin semilla (todo borde): se quedan como instancia propia
        orphan = mask & (lab == 0)
        if orphan.any():
            extra, _ = ndi.label(orphan)
            lab[orphan] = extra[orphan] + lab.max()
    else:
        lab = comps
    if cfg.merge_dist > 0:
        lab = _merge_fragments(lab, comps, cfg.merge_dist)
    return _filter(lab, pf, cfg)


def _merge_fragments(lab: np.ndarray, comps: np.ndarray, dist: int) -> np.ndarray:
    """Une segmentos de componentes distintas separadas por menos de `dist` px (union-find)."""
    ids = np.unique(lab)[1:]
    parent = {i: i for i in ids}

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * dist + 1, 2 * dist + 1))
    objs = ndi.find_objects(lab)
    for i in ids:
        sl = objs[i - 1]
        y0, y1 = max(sl[0].start - dist, 0), min(sl[0].stop + dist, lab.shape[0])
        x0, x1 = max(sl[1].start - dist, 0), min(sl[1].stop + dist, lab.shape[1])
        win = lab[y0:y1, x0:x1]
        own = win == i
        grown = cv2.dilate(own.astype(np.uint8), k).astype(bool)
        my_comp = comps[y0:y1, x0:x1][own][0]
        near = grown & (win > 0) & (win != i) & (comps[y0:y1, x0:x1] != my_comp)
        for j in np.unique(win[near]):
            parent[find(j)] = find(i)
    lut = np.zeros(lab.max() + 1, np.int32)
    for i in ids:
        lut[i] = find(i)
    return lut[lab]


def _filter(lab: np.ndarray, pf: np.ndarray, cfg: PostCfg) -> np.ndarray:
    ids, area = np.unique(lab, return_counts=True)
    score = ndi.mean(pf, lab, ids)
    keep = (ids > 0) & (area >= cfg.min_area) & (score >= cfg.min_score)
    lut = np.zeros(lab.max() + 1, np.int32)
    lut[ids[keep]] = np.arange(1, keep.sum() + 1)
    return lut[lab]

## 6. Candidate features and instance rescoring
Compute morphology, probability, contrast, disk position, year and
observatory features. The supplied histogram gradient-boosting models
score candidates. Keep candidates above the fitted threshold, retaining
at least the highest-scoring candidate when any exist.

In [ ]:
def candidates(prob: np.ndarray, cfg: PostCfg) -> np.ndarray:
    """Mapa de etiquetas con los candidatos: el post-proceso de `cfg` sin su filtro final."""
    return instances(prob, PostCfg(**{**cfg.as_dict(), "min_area": MIN_CANDIDATE, "min_score": 0.0}))


def disk(im: np.ndarray) -> tuple[np.ndarray, float]:
    """Centro (x, y) y radio del disco solar: la componente más grande por encima del fondo."""
    m = (cv2.GaussianBlur(im, (0, 0), 4) > 20).astype(np.uint8)
    _, _, stats, cents = cv2.connectedComponentsWithStats(m)
    k = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
    return cents[k], float(np.sqrt(stats[k, cv2.CC_STAT_AREA] / np.pi))


def features(prob: np.ndarray, im: np.ndarray, lab: np.ndarray, stem: str) -> pd.DataFrame:
    """Una fila por candidato de `lab` (ids 1..n, en orden)."""
    (cx, cy), radius = disk(im)
    pf, pb = prob[0].astype(np.float32), prob[1].astype(np.float32)
    ring_k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
    n = int(lab.max())
    rows = []
    for i, sl in enumerate(ndi.find_objects(lab), start=1):
        if sl is None:
            continue
        pad = 10
        y0, y1 = max(sl[0].start - pad, 0), min(sl[0].stop + pad, lab.shape[0])
        x0, x1 = max(sl[1].start - pad, 0), min(sl[1].stop + pad, lab.shape[1])
        m = lab[y0:y1, x0:x1] == i
        ring = cv2.dilate(m.astype(np.uint8), ring_k).astype(bool) & (lab[y0:y1, x0:x1] == 0)
        p, c, g = pf[y0:y1, x0:x1][m], pb[y0:y1, x0:x1][m], im[y0:y1, x0:x1].astype(np.float32)
        ys, xs = np.nonzero(m)
        (_, _), (w, h), _ = cv2.minAreaRect(np.stack([xs, ys], 1).astype(np.float32))
        contours, _ = cv2.findContours(m.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        perim = sum(cv2.arcLength(ct, True) for ct in contours)
        inside, around = g[m].mean(), g[ring].mean() if ring.any() else g[m].mean()
        r = np.hypot(xs.mean() + x0 - cx, ys.mean() + y0 - cy) / radius
        rows.append({
            "id": i, "area": m.sum(), "log_area": np.log(m.sum()),
            "p_mean": p.mean(), "p_max": p.max(), "p_p10": np.percentile(p, 10), "p_p90": np.percentile(p, 90),
            "p_frac_hi": (p > 0.8).mean(), "contact_mean": c.mean(), "contact_max": c.max(),
            "ring_p_mean": pf[y0:y1, x0:x1][ring].mean() if ring.any() else 0.0,
            "elong": max(w, h) / max(min(w, h), 1.0), "length": max(w, h), "width": m.sum() / max(max(w, h), 1.0),
            "compact": 4 * np.pi * m.sum() / max(perim, 1.0) ** 2, "n_parts": len(contours),
            "int_in": inside, "int_ring": around, "contrast": (around - inside) / max(around, 1.0),
            "int_std": g[m].std(), "r": r, "limb": float(r > 0.85),
        })
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    df["n_cand"] = n
    df["rank_area"] = df.area.rank(ascending=False, pct=True)
    df["rank_p"] = df.p_mean.rank(ascending=False, pct=True)
    df["disk_int"] = float(np.median(im[im > 20])) if (im > 20).any() else 0.0
    df["year"] = int(stem[:4])
    for o in OBSERVATORIES:
        df[f"obs_{o}"] = float(stem[-2] == o)
    return df


def keep(lab: np.ndarray, scores: np.ndarray, threshold: float, min_keep: int = 1) -> np.ndarray:
    """Conserva los candidatos con score >= threshold (al menos `min_keep`, los de mayor score) y renumera."""
    order = np.argsort(-scores)
    kept = scores >= threshold
    kept[order[:min_keep]] = True
    lut = np.zeros(lab.max() + 1, np.int32)
    ids = np.nonzero(kept)[0] + 1
    lut[ids] = np.arange(1, len(ids) + 1)
    return lut[lab]

## 7. CSV encoding and structural audit
Encode disjoint masks as compressed COCO RLE. Check column names,
image coverage, unique instance IDs, nonempty masks and zero overlaps.

In [ ]:
def build_submission(preds: dict[str, list[np.ndarray]]) -> pd.DataFrame:
    """preds: image_id (sin .jpeg) -> lista de máscaras binarias 2048x2048."""
    rows = []
    for img, masks in preds.items():
        for k, m in enumerate((m for m in masks if m.any()), start=1):
            rows.append((f"{img}_{k}", encode_counts(m)))
    return pd.DataFrame(rows, columns=["filament_id", "segmentation_rle"])


def validate(df: pd.DataFrame, test_dir: Path) -> list[str]:
    errors = []
    expected = {p.stem for p in Path(test_dir).glob("*.jpeg")}
    if list(df.columns) != ["filament_id", "segmentation_rle"]:
        errors.append(f"columnas inesperadas: {list(df.columns)}")
    if df["filament_id"].duplicated().any():
        errors.append("filament_id duplicados")
    present = {f.rsplit("_", 1)[0] for f in df["filament_id"]}
    if missing := expected - present:
        errors.append(f"{len(missing)} imágenes de test sin predicción")
    if extra := present - expected:
        errors.append(f"{len(extra)} ids que no son de test: {sorted(extra)[:3]}")
    for fid, counts in zip(df["filament_id"], df["segmentation_rle"]):
        try:
            m = decode_counts(counts)
        except Exception as e:  # noqa: BLE001
            errors.append(f"{fid}: RLE inválido ({e})")
            continue
        if m.shape != (SIZE, SIZE):
            errors.append(f"{fid}: forma {m.shape}")
        elif not m.any():
            errors.append(f"{fid}: máscara vacía")
    return errors

In [ ]:
def audit(path, test_dir):
    import numpy as np
    import pandas as pd
    frame = pd.read_csv(path)
    errors = validate(frame, test_dir)
    stems = frame.filament_id.str.rsplit("_", n=1).str[0]
    overlap_pixels = 0
    for stem, group in frame.groupby(stems):
        occupied = np.zeros((2048, 2048), bool)
        for encoded in group.segmentation_rle:
            mask = decode_counts(encoded).astype(bool)
            overlap_pixels += int((occupied & mask).sum())
            occupied |= mask
    if overlap_pixels:
        errors.append(f"Overlapping masks: {overlap_pixels} repeated pixels")
    return dict(csv=str(path), sha256=hashlib.sha256(Path(path).read_bytes()).hexdigest(),
                rows=len(frame), observations=stems.nunique(), overlapping_pixels=overlap_pixels,
                errors=errors, passed=not errors)

## 8. Load weights and run fresh inference
Both rescorers share the same five-model inference pass. Every image is
inferred afresh: the notebook does not reuse cached predictions. It writes
outputs only after model inference, then compares with the accepted CSVs.
CPU inference can take several minutes per image.

In [ ]:
def reproduce_predictions(device=DEVICE, limit=LIMIT):
    torch.set_num_threads(2)
    device = pick(device)
    paths = sorted(TEST_IMAGES.glob("*.jpeg"))
    assert len(paths) == 180, "Expected 180 official test images"
    if limit:
        paths = paths[:limit]
    judges = {key:joblib.load(ROOT / "runs/ens5c" / sub / "model.joblib")
              for key,sub in [("match","rescore"),("eiou","rescore_expected_iou")]}
    models = []
    for member in MEMBERS:
        folder = ROOT / "runs" / member
        net = build(json.loads((folder / "config.json").read_text()), pretrained=False)
        net.load_state_dict(torch.load(folder / "best.pt", map_location="cpu", weights_only=True))
        models.append(net.to(device).eval())
    outputs = OUTPUTS
    outputs.mkdir(parents=True, exist_ok=True)
    parts = {key:[] for key in judges}
    for n, path in enumerate(paths, 1):
        image = read_image(path)
        probs = [predict(model, image, tta=True, device=device).astype(np.float32) for model in models]
        probability = np.mean(probs, axis=0)
        rows = {}
        for variant, judge in judges.items():
            cfg = PostCfg(**judge["cfg"])
            labels = candidates(probability, cfg)
            feats = features(probability, image, labels, path.stem)
            scores = judge["model"].predict(feats[judge["features"]].values) if len(feats) else np.zeros(0)
            labels = keep(labels, scores, judge["threshold"], min_keep=1)
            if labels.max() == 0:
                loose = instances(probability, PostCfg(t_fg=cfg.t_fg, t_b=cfg.t_b, split=cfg.split,
                                                      merge_dist=cfg.merge_dist, min_area=50))
                if loose.max():
                    labels = (loose == np.bincount(loose.ravel())[1:].argmax()+1).astype(np.int32)
            frame = build_submission({path.stem:[labels == k for k in range(1, int(labels.max())+1)]})
            rows[variant] = frame.to_dict("records")
        for variant in judges:
            parts[variant].extend(rows[variant])
        print(f"{n}/{len(paths)} {path.stem}", flush=True)
    report = dict(full_dataset=not limit, images=len(paths), device=device, torch=torch.__version__,
                  checkpoint_manifest_sha256=hashlib.sha256((ROOT / "weights/manifest.json").read_bytes()).hexdigest(),
                  reference_used_for_inference=False, variants={})
    for variant in judges:
        filename = f"submission_ens5c_rescore_{variant}.csv"
        target = outputs / (f"smoke_{filename}" if limit else filename)
        frame = pd.DataFrame(parts[variant], columns=["filament_id","segmentation_rle"])
        frame.to_csv(target,index=False)
        reference = pd.read_csv(ROOT / "reference_predictions" / filename)
        allowed = {p.stem for p in paths}
        reference = reference[reference.filament_id.str.rsplit("_",n=1).str[0].isin(allowed)]
        a = frame.filament_id.str.rsplit("_",n=1).str[0]
        b = reference.filament_id.str.rsplit("_",n=1).str[0]
        total_iou = tp = fp = fn = 0
        for stem in sorted(allowed):
            masks_a = [dict(size=[2048,2048],counts=c.encode()) for c in frame.loc[a==stem,"segmentation_rle"]]
            masks_b = [dict(size=[2048,2048],counts=c.encode()) for c in reference.loc[b==stem,"segmentation_rle"]]
            if masks_a and masks_b:
                iou = mask_util.iou(masks_a,masks_b,[0]*len(masks_b))
                hit = iou > .5
                total_iou += float(iou[hit].sum()); tp += int(hit.sum())
                fp += int((hit.sum(1)==0).sum()); fn += int((hit.sum(0)==0).sum())
            else:
                fp += len(masks_a); fn += len(masks_b)
        report["variants"][variant] = dict(rows=len(frame), sha256=hashlib.sha256(target.read_bytes()).hexdigest(),
            exact_rows=frame.reset_index(drop=True).equals(reference.reset_index(drop=True)),
            prediction_agreement_pq=total_iou/max(tp+.5*(fp+fn),1),
            agreement_note="Comparison with submitted predictions, not test accuracy")
    output = outputs / ("smoke_reproduction_report.json" if limit else "reproduction_report.json")
    output.write_text(json.dumps(report,indent=2))
    print(json.dumps(report,indent=2),flush=True)

    return report

In [ ]:
reproduction_report = reproduce_predictions()

## 9. Audit results
For a full run, every official test image must be represented. For a smoke
run, require exactly the selected subset and label the audit as partial.
Prediction-agreement PQ compares two prediction files; it is **not**
accuracy against the hidden test labels. Hardware/library differences
can affect contour pixels. The original 180-image CUDA reproduction is
recorded separately in `metadata/full_reproduction.json`.

In [ ]:
audit_results = {}
for variant in ('match', 'eiou'):
    prefix = 'smoke_' if LIMIT else ''
    target = OUTPUTS / f'{prefix}submission_ens5c_rescore_{variant}.csv'
    if LIMIT:
        # The original audit expects a directory of expected image IDs.
        # Temporary empty files specify the smoke subset, not image data.
        import tempfile
        with tempfile.TemporaryDirectory() as temporary:
            subset = Path(temporary)
            for path in test_paths[:LIMIT]:
                (subset / path.name).touch()
            result = audit(target, subset)
    else:
        result = audit(target, TEST_IMAGES)
    result['full_dataset'] = not bool(LIMIT)
    (OUTPUTS / f'{prefix}{variant}_audit.json').write_text(json.dumps(result, indent=2))
    assert result['passed'], result['errors']
    audit_results[variant] = result
print(json.dumps({'reproduction': reproduction_report, 'audits': audit_results}, indent=2))

## Evaluation and training evidence
The report describes the training procedure, annotation disagreement,
morphology and limitations. `runs/*/config.json` and `log.csv` preserve
training settings and histories. Training and fitting from scratch remain
available as separate research scripts; they are not dependencies of this
notebook. Local nested rescorer validation reused a development split
previously used to select segmenters and post-processing, so it is not
an independent estimate of the complete model-selection process.